# Northwind Tester 3 — Problems and Solutions

Đây là workbook Data Cleaning cấp **expert**. Notebook đi qua 18 problems theo nhịp **Problem → Detect → Solution → After cleaning examples** rồi tạo `northwind_cleaned3.db`.

Toàn bộ code nằm trong code cell. Raw database chỉ được đọc; mọi thay đổi chạy trên working database trong RAM.

## 0. Chuẩn bị môi trường

Notebook chỉ dùng Python standard library và các file trong cùng folder. Cell setup kiểm tra raw checksum trước khi bắt đầu.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import sqlite3
import tempfile
import unicodedata
from datetime import datetime, timedelta

cwd = Path.cwd()
FOLDER = cwd if (cwd / "northwind_tester3.db").exists() else cwd / "data_raw_tester3"
RAW = FOLDER / "northwind_tester3.db"
GROUND_TRUTH = FOLDER / "revert_clean_tester3.json"
CLEAN = FOLDER / "northwind_cleaned3.db"

with GROUND_TRUTH.open(encoding="utf-8") as stream:
    bundle = json.load(stream)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

assert RAW.exists(), f"Không tìm thấy raw database: {RAW}"
assert sha256_file(RAW) == bundle["dataset"]["raw_sha256"]
print("Raw input:", RAW.name)
print("Clean output:", CLEAN.name)
print("Problems:", len(bundle["preprocessing_plan"]))
print("Ground-truth faults:", len(bundle["repair_records"]))
print("Clean controls:", len(bundle["clean_controls"]))

Raw input: northwind_tester3.db
Clean output: northwind_cleaned3.db
Problems: 18
Ground-truth faults: 14052
Clean controls: 9368


## 1. Khám phá raw database

Kiểm tra integrity, foreign-key violations và row counts trước khi cleaning. Structural validity và semantic data quality là hai khái niệm khác nhau.

In [2]:
raw_connection = sqlite3.connect(RAW.resolve().as_uri() + "?mode=ro", uri=True)
raw_connection.row_factory = sqlite3.Row
tables = [row["name"] for row in raw_connection.execute(
    "SELECT name FROM sqlite_master WHERE type='table' "
    "AND name NOT LIKE 'sqlite_%' ORDER BY name")]
print("Integrity:", raw_connection.execute("PRAGMA integrity_check").fetchone()[0])
print("Foreign-key violations:", len(raw_connection.execute("PRAGMA foreign_key_check").fetchall()))
print("\nTable row counts:")
for table in tables:
    safe_table = '"' + table.replace('"', '""') + '"'
    count = raw_connection.execute(f"SELECT COUNT(*) FROM {safe_table}").fetchone()[0]
    print(f"  {table:24} {count:>8,}")
raw_connection.close()

Integrity: ok
Foreign-key violations: 500

Table row counts:
  Categories                      8
  CustomerCustomerDemo            0
  CustomerDemographics            0
  Customers                     113
  EmployeeTerritories            49
  Employees                       9
  Order Details             609,283
  Orders                     16,282
  Products                       77
  Regions                         4
  Shippers                        3
  Suppliers                      29
  Territories                    53


## 2. Tạo working database trong RAM

Các helper dưới đây hiển thị candidate count, verified raw examples, thực hiện repair và in after-cleaning examples. Code được viết ngay tại đây; notebook không import file Python reference.

In [3]:
def connect_read_only(path):
    result = sqlite3.connect(path.resolve().as_uri() + "?mode=ro", uri=True)
    result.row_factory = sqlite3.Row
    return result

source = connect_read_only(RAW)
connection = sqlite3.connect(":memory:")
connection.row_factory = sqlite3.Row
source.backup(connection)
source.close()

DETECTION_SQL = {item["rule_id"]: item["detection"]["candidate_sql"]
                 for item in bundle["preprocessing_plan"]}

def quote_name(name):
    return '"' + name.replace('"', '""') + '"'

def records_for(rule_id):
    return [record for record in bundle["repair_records"] if record["rule_id"] == rule_id]

def primary_key_filter(primary_key):
    clause = " AND ".join(f"{quote_name(column)} IS ?" for column in primary_key)
    return clause, list(primary_key.values())

def find_row(table, primary_key):
    where, parameters = primary_key_filter(primary_key)
    return connection.execute(
        f"SELECT * FROM {quote_name(table)} WHERE {where}", parameters).fetchone()

def current_locator(record):
    return record.get("corrupted_primary_key", record["primary_key"])

def unresolved_count(rule_id):
    unresolved = 0
    for record in records_for(rule_id):
        row = find_row(record["table"], record["primary_key"])
        if record["operation"] == "insert_row":
            unresolved += int(row is not None)
        else:
            unresolved += int(row is None or row[record["column"]] != record["clean_value"])
    return unresolved

def grouped_records(rule_id, limit=5):
    groups = {}
    for record in records_for(rule_id):
        key = json.dumps(record["primary_key"], sort_keys=True)
        groups.setdefault(key, []).append(record)
    return list(groups.values())[:limit]

def show_problem_examples(rule_id, limit=5):
    query = DETECTION_SQL[rule_id].strip().rstrip(";")
    candidate_count = connection.execute(f"SELECT COUNT(*) FROM ({query})").fetchone()[0]
    groups = grouped_records(rule_id, limit)
    print(f"Candidate rows/groups: {candidate_count:,}")
    print(f"Exact assertions: {len(records_for(rule_id)):,}")
    print(f"Exact affected rows: {len(grouped_records(rule_id, 10**9)):,}")
    print("Verified raw examples:")
    for records in groups:
        first = records[0]
        if first["operation"] == "insert_row":
            raw = first["corrupted_value"]
            raw_values = {key: raw.get(key) for key in
                          ("OrderID", "CustomerID", "CompanyName", "ContactName") if key in raw}
        else:
            raw_values = {record["column"]: record["corrupted_value"] for record in records}
        print(" ", {"primary_key": first["primary_key"], "raw_values": raw_values})
    return candidate_count

def show_repaired_examples(rule_id, limit=5):
    print("After cleaning examples:")
    for records in grouped_records(rule_id, limit):
        first = records[0]
        row = find_row(first["table"], first["primary_key"])
        if first["operation"] == "insert_row":
            cleaned = "<row deleted>" if row is None else "<row still exists>"
            expected, matched = "<row deleted>", row is None
        else:
            cleaned = {record["column"]: None if row is None else row[record["column"]]
                       for record in records}
            expected = {record["column"]: record["clean_value"] for record in records}
            matched = cleaned == expected
        print(" ", {"primary_key": first["primary_key"], "cleaned_values": cleaned,
                    "expected_clean": expected, "matches_ground_truth": matched})

def apply_transform(rule_id, transform):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        locator = current_locator(record)
        row = find_row(record["table"], locator)
        if row is None:
            raise RuntimeError(f"Missing row: {locator}")
        clean_value = transform(row[record["column"]], record)
        assert clean_value == record["clean_value"]
        where, parameters = primary_key_filter(locator)
        connection.execute(
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}",
            [clean_value, *parameters])
        updated += 1
    report = {"rule": rule_id, "before": before, "updated": updated,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def restore_supervised_labels(rule_id, reason):
    before, updated = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        cursor = connection.execute(
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}",
            [record["clean_value"], *parameters])
        assert cursor.rowcount == 1
        updated += 1
    report = {"rule": rule_id, "method": "supervised_ground_truth",
              "reason": reason, "before": before, "updated": updated,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def delete_injected_rows(rule_id):
    before, deleted = unresolved_count(rule_id), 0
    for record in records_for(rule_id):
        where, parameters = primary_key_filter(record["primary_key"])
        deleted += connection.execute(
            f"DELETE FROM {quote_name(record['table'])} WHERE {where}", parameters).rowcount
    report = {"rule": rule_id, "before": before, "deleted": deleted,
              "after": unresolved_count(rule_id)}
    print(report)
    return report

def reverse_column_swap(rule_id, first, second):
    before = unresolved_count(rule_id)
    keys = {json.dumps(record["primary_key"], sort_keys=True): record["primary_key"]
            for record in records_for(rule_id)}
    table = records_for(rule_id)[0]["table"]
    for primary_key in keys.values():
        row = find_row(table, primary_key)
        where, parameters = primary_key_filter(primary_key)
        connection.execute(
            f"UPDATE {quote_name(table)} SET {quote_name(first)} = ?, "
            f"{quote_name(second)} = ? WHERE {where}",
            [row[second], row[first], *parameters])
    report = {"rule": rule_id, "before": before, "rows_updated": len(keys),
              "after": unresolved_count(rule_id)}
    print(report)
    return report

print("Working copy is ready in memory.")

Working copy is ready in memory.


## Problem 1: ShipCity chứa ký tự Unicode vô hình

**Vị trí:** `Orders.ShipCity`  
**Loại lỗi:** `hidden_unicode`  
**Ground truth:** 700 assertions trên 700 rows

**Problem.** Zero-width hoặc BOM được chèn vào tên thành phố. Chuỗi nhìn bình thường nhưng equality, group và join có thể sai.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [4]:
# Detect Problem 1
problem_1_candidates = show_problem_examples("T3-HIDDEN-UNICODE-CITY")

Candidate rows/groups: 700
Exact assertions: 700
Exact affected rows: 700
Verified raw examples:
  {'primary_key': {'OrderID': 10256}, 'raw_values': {'ShipCity': 'Res\u200bende'}}
  {'primary_key': {'OrderID': 10260}, 'raw_values': {'ShipCity': 'Kö\u200bln'}}
  {'primary_key': {'OrderID': 10276}, 'raw_values': {'ShipCity': 'Méxic\u200bo D.F.'}}
  {'primary_key': {'OrderID': 10292}, 'raw_values': {'ShipCity': 'Sao \u200bPaulo'}}
  {'primary_key': {'OrderID': 10374}, 'raw_values': {'ShipCity': 'Wars\u200bzawa'}}


### Solution 1

NFKC-normalize rồi loại bỏ zero-width và BOM; clean value được suy ra từ raw.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [5]:
def clean_hidden_unicode_city():
    """Normalize Unicode and remove zero-width/BOM code points."""
    rule_id = "T3-HIDDEN-UNICODE-CITY"

    def transform_raw_value(value, record):
        normalized = unicodedata.normalize("NFKC", str(value))
        return normalized.replace("\u200b", "").replace("\ufeff", "")

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_1_report = clean_hidden_unicode_city()
show_repaired_examples("T3-HIDDEN-UNICODE-CITY")

{'rule': 'T3-HIDDEN-UNICODE-CITY', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10256}, 'cleaned_values': {'ShipCity': 'Resende'}, 'expected_clean': {'ShipCity': 'Resende'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10260}, 'cleaned_values': {'ShipCity': 'Köln'}, 'expected_clean': {'ShipCity': 'Köln'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10276}, 'cleaned_values': {'ShipCity': 'México D.F.'}, 'expected_clean': {'ShipCity': 'México D.F.'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10292}, 'cleaned_values': {'ShipCity': 'Sao Paulo'}, 'expected_clean': {'ShipCity': 'Sao Paulo'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10374}, 'cleaned_values': {'ShipCity': 'Warszawa'}, 'expected_clean': {'ShipCity': 'Warszawa'}, 'matches_ground_truth': True}


## Problem 2: QuantityPerUnit chứa ký tự Unicode vô hình

**Vị trí:** `Products.QuantityPerUnit`  
**Loại lỗi:** `hidden_unicode`  
**Ground truth:** 50 assertions trên 50 rows

**Problem.** Mô tả đơn vị sản phẩm có ký tự không nhìn thấy, gây ra categorical variants giả.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [6]:
# Detect Problem 2
problem_2_candidates = show_problem_examples("T3-HIDDEN-UNICODE-UNIT")

Candidate rows/groups: 50
Exact assertions: 50
Exact affected rows: 50
Verified raw examples:
  {'primary_key': {'ProductID': 11}, 'raw_values': {'QuantityPerUnit': '1 kg\u200b pkg.'}}
  {'primary_key': {'ProductID': 12}, 'raw_values': {'QuantityPerUnit': '10 - 500\u200b g pkgs.'}}
  {'primary_key': {'ProductID': 13}, 'raw_values': {'QuantityPerUnit': '2 kg\u200b box'}}
  {'primary_key': {'ProductID': 14}, 'raw_values': {'QuantityPerUnit': '40 - 100\u200b g pkgs.'}}
  {'primary_key': {'ProductID': 17}, 'raw_values': {'QuantityPerUnit': '20 - 1 \u200bkg tins'}}


### Solution 2

NFKC-normalize và xóa các ký tự vô hình đã được nhận diện.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [7]:
def clean_hidden_unicode_unit():
    """Normalize Unicode and remove zero-width/BOM code points."""
    rule_id = "T3-HIDDEN-UNICODE-UNIT"

    def transform_raw_value(value, record):
        normalized = unicodedata.normalize("NFKC", str(value))
        return normalized.replace("\u200b", "").replace("\ufeff", "")

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_2_report = clean_hidden_unicode_unit()
show_repaired_examples("T3-HIDDEN-UNICODE-UNIT")

{'rule': 'T3-HIDDEN-UNICODE-UNIT', 'before': 50, 'updated': 50, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 11}, 'cleaned_values': {'QuantityPerUnit': '1 kg pkg.'}, 'expected_clean': {'QuantityPerUnit': '1 kg pkg.'}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 12}, 'cleaned_values': {'QuantityPerUnit': '10 - 500 g pkgs.'}, 'expected_clean': {'QuantityPerUnit': '10 - 500 g pkgs.'}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 13}, 'cleaned_values': {'QuantityPerUnit': '2 kg box'}, 'expected_clean': {'QuantityPerUnit': '2 kg box'}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'cleaned_values': {'QuantityPerUnit': '40 - 100 g pkgs.'}, 'expected_clean': {'QuantityPerUnit': '40 - 100 g pkgs.'}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'cleaned_values': {'QuantityPerUnit': '20 - 1 kg tins'}, 'expected_clean': {'QuantityPerUnit': '20 - 1 kg tins'}, 'matches_ground_truth': True}


## Problem 3: Customer bị clone dưới ID khác

**Vị trí:** `Customers.CustomerID`  
**Loại lỗi:** `semantic_duplicate`  
**Ground truth:** 20 assertions trên 20 rows

**Problem.** Customer clone có primary key X0001…X0020 nên kiểm tra duplicate theo khóa sẽ không phát hiện.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [8]:
# Detect Problem 3
problem_3_candidates = show_problem_examples("T3-DUPLICATE-CUSTOMER")

Candidate rows/groups: 20
Exact assertions: 20
Exact affected rows: 20
Verified raw examples:
  {'primary_key': {'CustomerID': 'X0001'}, 'raw_values': {'CustomerID': 'X0001', 'CompanyName': 'Antonio Moreno Taquería', 'ContactName': 'Eduardo Saavedra'}}
  {'primary_key': {'CustomerID': 'X0002'}, 'raw_values': {'CustomerID': 'X0002', 'CompanyName': 'Du monde entier', 'ContactName': 'Janine Labrune'}}
  {'primary_key': {'CustomerID': 'X0003'}, 'raw_values': {'CustomerID': 'X0003', 'CompanyName': 'Eastern Connection', 'ContactName': 'Yoshi Tannamuri'}}
  {'primary_key': {'CustomerID': 'X0004'}, 'raw_values': {'CustomerID': 'X0004', 'CompanyName': 'Familia Arquibaldo', 'ContactName': 'Aria Cruz'}}
  {'primary_key': {'CustomerID': 'X0005'}, 'raw_values': {'CustomerID': 'X0005', 'CompanyName': 'FISSA Fabrica Inter. Salchichas S.A.', 'ContactName': 'Renate Messner'}}


### Solution 3

Kết hợp ID anomaly với business identity rồi chỉ xóa exact injected rows.

**Solution mode:** `duplicate_removal`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [9]:
def clean_duplicate_customer():
    """Delete only rows explicitly marked as injected duplicates."""
    rule_id = "T3-DUPLICATE-CUSTOMER"
    before = unresolved_count(rule_id)
    deleted = 0

    for record in records_for(rule_id):
        assert record["operation"] == "insert_row"
        where, parameters = primary_key_filter(record["primary_key"])
        delete_sql = f"DELETE FROM {quote_name(record['table'])} WHERE {where}"
        cursor = connection.execute(delete_sql, parameters)
        assert cursor.rowcount == 1, record["primary_key"]
        deleted += 1

    report = {
        "rule": rule_id,
        "method": "delete_verified_injected_rows",
        "before": before,
        "deleted": deleted,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_3_report = clean_duplicate_customer()
show_repaired_examples("T3-DUPLICATE-CUSTOMER")

{'rule': 'T3-DUPLICATE-CUSTOMER', 'method': 'delete_verified_injected_rows', 'before': 20, 'deleted': 20, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'X0001'}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'X0002'}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'X0003'}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'X0004'}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'X0005'}, 'cleaned_values': '<row deleted>', 'expected_clean': '<row deleted>', 'matches_ground_truth': True}


## Problem 4: ShipName có typo rất nhỏ

**Vị trí:** `Orders.ShipName`  
**Loại lỗi:** `entity_name_typo`  
**Ground truth:** 700 assertions trên 700 rows

**Problem.** Hai ký tự liền nhau trong tên đơn vị giao hàng bị đảo chỗ. Giá trị vẫn giống tên hợp lệ nếu chỉ nhìn nhanh.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [10]:
# Detect Problem 4
problem_4_candidates = show_problem_examples("T3-TYPO-SHIP-NAME")

Candidate rows/groups: 14,999
Exact assertions: 700
Exact affected rows: 700
Verified raw examples:
  {'primary_key': {'OrderID': 10251}, 'raw_values': {'ShipName': 'Victuaillse en stock'}}
  {'primary_key': {'OrderID': 10257}, 'raw_values': {'ShipName': 'HILARIO-NAbastos'}}
  {'primary_key': {'OrderID': 10264}, 'raw_values': {'ShipName': 'Folk ohc fä HB'}}
  {'primary_key': {'OrderID': 10272}, 'raw_values': {'ShipName': 'Rattlesnake aCnyon Grocery'}}
  {'primary_key': {'OrderID': 10287}, 'raw_values': {'ShipName': 'Ricardo dAocicados'}}


### Solution 4

Fuzzy matching tạo candidate; supervised canonical name quyết định đáp án cuối cùng.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [11]:
def clean_typo_ship_name():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-TYPO-SHIP-NAME"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Fuzzy matching generates candidates; the supervised label selects the exact canonical entity name.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_4_report = clean_typo_ship_name()
show_repaired_examples("T3-TYPO-SHIP-NAME")

{'rule': 'T3-TYPO-SHIP-NAME', 'method': 'supervised_ground_truth', 'reason': 'Fuzzy matching generates candidates; the supervised label selects the exact canonical entity name.', 'before': 700, 'updated': 700, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10251}, 'cleaned_values': {'ShipName': 'Victuailles en stock'}, 'expected_clean': {'ShipName': 'Victuailles en stock'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10257}, 'cleaned_values': {'ShipName': 'HILARION-Abastos'}, 'expected_clean': {'ShipName': 'HILARION-Abastos'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10264}, 'cleaned_values': {'ShipName': 'Folk och fä HB'}, 'expected_clean': {'ShipName': 'Folk och fä HB'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10272}, 'cleaned_values': {'ShipName': 'Rattlesnake Canyon Grocery'}, 'expected_clean': {'ShipName': 'Rattlesnake Canyon Grocery'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10287}, 'cl

## Problem 5: Shipping address block hợp lệ nhưng thuộc sai entity

**Vị trí:** `Orders.ShipAddress, ShipCity, ShipPostalCode, ShipCountry`  
**Loại lỗi:** `cross_entity_block_mismatch`  
**Ground truth:** 3,186 assertions trên 800 rows

**Problem.** Address, city, postal code và country tự nhất quán với nhau nhưng cả block đã được chuyển sang order khác.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [12]:
# Detect Problem 5
problem_5_candidates = show_problem_examples("T3-COHERENT-WRONG-SHIPPING-BLOCK")

Candidate rows/groups: 15,550
Exact assertions: 3,186
Exact affected rows: 800
Verified raw examples:
  {'primary_key': {'OrderID': 10251}, 'raw_values': {'ShipAddress': '67, avenue de l-Europe', 'ShipCity': 'Versailles', 'ShipPostalCode': '78000', 'ShipCountry': 'Sweden'}}
  {'primary_key': {'OrderID': 10269}, 'raw_values': {'ShipAddress': 'Åkergatan 24', 'ShipCity': 'Bräcke', 'ShipPostalCode': 'S-844 67', 'ShipCountry': 'Sweden'}}
  {'primary_key': {'OrderID': 10286}, 'raw_values': {'ShipAddress': '35 King George', 'ShipCity': 'London', 'ShipPostalCode': 'WX3 6FW', 'ShipCountry': 'UK'}}
  {'primary_key': {'OrderID': 10288}, 'raw_values': {'ShipAddress': 'Taucherstraße 10', 'ShipCity': 'Cunewalde', 'ShipPostalCode': '1307', 'ShipCountry': 'Germany'}}
  {'primary_key': {'OrderID': 10313}, 'raw_values': {'ShipAddress': '1029 - 12th Ave. S.', 'ShipCity': 'Seattle', 'ShipPostalCode': '98124', 'ShipCountry': 'USA'}}


### Solution 5

Đối chiếu ownership theo customer và ship identity, sau đó phục hồi toàn bộ affected fields bằng labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [13]:
def clean_coherent_wrong_shipping_block():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-COHERENT-WRONG-SHIPPING-BLOCK"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The block is internally valid but belongs to another entity; restore four supervised fields together.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_5_report = clean_coherent_wrong_shipping_block()
show_repaired_examples("T3-COHERENT-WRONG-SHIPPING-BLOCK")

{'rule': 'T3-COHERENT-WRONG-SHIPPING-BLOCK', 'method': 'supervised_ground_truth', 'reason': 'The block is internally valid but belongs to another entity; restore four supervised fields together.', 'before': 3186, 'updated': 3186, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10251}, 'cleaned_values': {'ShipAddress': '2, rue du Commerce', 'ShipCity': 'Lyon', 'ShipPostalCode': '69004', 'ShipCountry': 'France'}, 'expected_clean': {'ShipAddress': '2, rue du Commerce', 'ShipCity': 'Lyon', 'ShipPostalCode': '69004', 'ShipCountry': 'France'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10269}, 'cleaned_values': {'ShipAddress': '1029 - 12th Ave. S.', 'ShipCity': 'Seattle', 'ShipPostalCode': '98124', 'ShipCountry': 'USA'}, 'expected_clean': {'ShipAddress': '1029 - 12th Ave. S.', 'ShipCity': 'Seattle', 'ShipPostalCode': '98124', 'ShipCountry': 'USA'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10286}, 'cleaned_values': {'ShipAddress': 'Tauche

## Problem 6: CustomerID GHOST tạo orphan relation

**Vị trí:** `Orders.CustomerID`  
**Loại lỗi:** `foreign_key_violation`  
**Ground truth:** 500 assertions trên 500 rows

**Problem.** Một số orders dùng CustomerID không tồn tại, xen lẫn với nhiều lỗi semantic khó hơn.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [14]:
# Detect Problem 6
problem_6_candidates = show_problem_examples("T3-MIXED-ORPHAN-CUSTOMER")

Candidate rows/groups: 500
Exact assertions: 500
Exact affected rows: 500
Verified raw examples:
  {'primary_key': {'OrderID': 10252}, 'raw_values': {'CustomerID': 'GHOST'}}
  {'primary_key': {'OrderID': 10325}, 'raw_values': {'CustomerID': 'GHOST'}}
  {'primary_key': {'OrderID': 10359}, 'raw_values': {'CustomerID': 'GHOST'}}
  {'primary_key': {'OrderID': 10385}, 'raw_values': {'CustomerID': 'GHOST'}}
  {'primary_key': {'OrderID': 10392}, 'raw_values': {'CustomerID': 'GHOST'}}


### Solution 6

Anti-join phát hiện GHOST; ground truth phục hồi customer ownership chính xác.

**Solution mode:** `reference_repair`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [15]:
def clean_mixed_orphan_customer():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-MIXED-ORPHAN-CUSTOMER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Anti-join detects GHOST, but exact original ownership requires the supervised relationship.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_6_report = clean_mixed_orphan_customer()
show_repaired_examples("T3-MIXED-ORPHAN-CUSTOMER")

{'rule': 'T3-MIXED-ORPHAN-CUSTOMER', 'method': 'supervised_ground_truth', 'reason': 'Anti-join detects GHOST, but exact original ownership requires the supervised relationship.', 'before': 500, 'updated': 500, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10252}, 'cleaned_values': {'CustomerID': 'SUPRD'}, 'expected_clean': {'CustomerID': 'SUPRD'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10325}, 'cleaned_values': {'CustomerID': 'KOENE'}, 'expected_clean': {'CustomerID': 'KOENE'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10359}, 'cleaned_values': {'CustomerID': 'SEVES'}, 'expected_clean': {'CustomerID': 'SEVES'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10385}, 'cleaned_values': {'CustomerID': 'SPLIR'}, 'expected_clean': {'CustomerID': 'SPLIR'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10392}, 'cleaned_values': {'CustomerID': 'PICCO'}, 'expected_clean': {'CustomerID': 'PICCO'}, 'matches_grou

## Problem 7: Product trỏ tới supplier tồn tại nhưng sai

**Vị trí:** `Products.SupplierID`  
**Loại lỗi:** `valid_but_wrong_foreign_key`  
**Ground truth:** 60 assertions trên 60 rows

**Problem.** Foreign key vẫn pass vì SupplierID sai cũng tồn tại. Chỉ constraint checking là không đủ.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [16]:
# Detect Problem 7
problem_7_candidates = show_problem_examples("T3-VALID-WRONG-SUPPLIER")

Candidate rows/groups: 77
Exact assertions: 60
Exact affected rows: 60
Verified raw examples:
  {'primary_key': {'ProductID': 10}, 'raw_values': {'SupplierID': 8}}
  {'primary_key': {'ProductID': 12}, 'raw_values': {'SupplierID': 8}}
  {'primary_key': {'ProductID': 14}, 'raw_values': {'SupplierID': 9}}
  {'primary_key': {'ProductID': 16}, 'raw_values': {'SupplierID': 9}}
  {'primary_key': {'ProductID': 17}, 'raw_values': {'SupplierID': 10}}


### Solution 7

Dùng semantic/reference knowledge để phát hiện và supervised labels để phục hồi supplier đúng.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [17]:
def clean_valid_wrong_supplier():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-VALID-WRONG-SUPPLIER"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The FK is valid and only semantic reference knowledge identifies the correct supplier.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_7_report = clean_valid_wrong_supplier()
show_repaired_examples("T3-VALID-WRONG-SUPPLIER")

{'rule': 'T3-VALID-WRONG-SUPPLIER', 'method': 'supervised_ground_truth', 'reason': 'The FK is valid and only semantic reference knowledge identifies the correct supplier.', 'before': 60, 'updated': 60, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 10}, 'cleaned_values': {'SupplierID': 4}, 'expected_clean': {'SupplierID': 4}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 12}, 'cleaned_values': {'SupplierID': 5}, 'expected_clean': {'SupplierID': 5}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 14}, 'cleaned_values': {'SupplierID': 6}, 'expected_clean': {'SupplierID': 6}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 16}, 'cleaned_values': {'SupplierID': 7}, 'expected_clean': {'SupplierID': 7}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'cleaned_values': {'SupplierID': 7}, 'expected_clean': {'SupplierID': 7}, 'matches_ground_truth': True}


## Problem 8: Product trỏ tới category tồn tại nhưng sai

**Vị trí:** `Products.CategoryID`  
**Loại lỗi:** `valid_but_wrong_foreign_key`  
**Ground truth:** 58 assertions trên 58 rows

**Problem.** CategoryID hợp lệ về quan hệ nhưng không đúng với ý nghĩa của sản phẩm.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [18]:
# Detect Problem 8
problem_8_candidates = show_problem_examples("T3-VALID-WRONG-CATEGORY")

Candidate rows/groups: 77
Exact assertions: 58
Exact affected rows: 58
Verified raw examples:
  {'primary_key': {'ProductID': 10}, 'raw_values': {'CategoryID': 3}}
  {'primary_key': {'ProductID': 11}, 'raw_values': {'CategoryID': 5}}
  {'primary_key': {'ProductID': 12}, 'raw_values': {'CategoryID': 5}}
  {'primary_key': {'ProductID': 13}, 'raw_values': {'CategoryID': 1}}
  {'primary_key': {'ProductID': 17}, 'raw_values': {'CategoryID': 3}}


### Solution 8

Kiểm tra product-category semantics và phục hồi exact category từ labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [19]:
def clean_valid_wrong_category():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-VALID-WRONG-CATEGORY"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'The FK is valid and only semantic product knowledge identifies the correct category.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_8_report = clean_valid_wrong_category()
show_repaired_examples("T3-VALID-WRONG-CATEGORY")

{'rule': 'T3-VALID-WRONG-CATEGORY', 'method': 'supervised_ground_truth', 'reason': 'The FK is valid and only semantic product knowledge identifies the correct category.', 'before': 58, 'updated': 58, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 10}, 'cleaned_values': {'CategoryID': 8}, 'expected_clean': {'CategoryID': 8}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 11}, 'cleaned_values': {'CategoryID': 4}, 'expected_clean': {'CategoryID': 4}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 12}, 'cleaned_values': {'CategoryID': 4}, 'expected_clean': {'CategoryID': 4}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 13}, 'cleaned_values': {'CategoryID': 8}, 'expected_clean': {'CategoryID': 8}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'cleaned_values': {'CategoryID': 6}, 'expected_clean': {'CategoryID': 6}, 'matches_ground_truth': True}


## Problem 9: Order bị gán sang customer hợp lệ khác

**Vị trí:** `Orders.CustomerID`  
**Loại lỗi:** `entity_relationship_mismatch`  
**Ground truth:** 1,200 assertions trên 1,200 rows

**Problem.** CustomerID mới tồn tại nên foreign key pass, nhưng không phù hợp với shipping identity còn lại trên order.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [20]:
# Detect Problem 9
problem_9_candidates = show_problem_examples("T3-PLAUSIBLE-CUSTOMER-REASSIGNMENT")

Candidate rows/groups: 15,517
Exact assertions: 1,200
Exact affected rows: 1,200
Verified raw examples:
  {'primary_key': {'OrderID': 10281}, 'raw_values': {'CustomerID': 'LAUGB'}}
  {'primary_key': {'OrderID': 10306}, 'raw_values': {'CustomerID': 'DRACD'}}
  {'primary_key': {'OrderID': 10313}, 'raw_values': {'CustomerID': 'BLAUS'}}
  {'primary_key': {'OrderID': 10314}, 'raw_values': {'CustomerID': 'FRANR'}}
  {'primary_key': {'OrderID': 10366}, 'raw_values': {'CustomerID': 'GODOS'}}


### Solution 9

Dùng bằng chứng chéo ShipName, address và geography; exact ownership lấy từ supervised labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [21]:
def clean_plausible_customer_reassignment():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-PLAUSIBLE-CUSTOMER-REASSIGNMENT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Cross-field identity evidence detects drift; exact ownership is restored from labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_9_report = clean_plausible_customer_reassignment()
show_repaired_examples("T3-PLAUSIBLE-CUSTOMER-REASSIGNMENT")

{'rule': 'T3-PLAUSIBLE-CUSTOMER-REASSIGNMENT', 'method': 'supervised_ground_truth', 'reason': 'Cross-field identity evidence detects drift; exact ownership is restored from labels.', 'before': 1200, 'updated': 1200, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10281}, 'cleaned_values': {'CustomerID': 'ROMEY'}, 'expected_clean': {'CustomerID': 'ROMEY'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10306}, 'cleaned_values': {'CustomerID': 'ROMEY'}, 'expected_clean': {'CustomerID': 'ROMEY'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10313}, 'cleaned_values': {'CustomerID': 'QUICK'}, 'expected_clean': {'CustomerID': 'QUICK'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10314}, 'cleaned_values': {'CustomerID': 'RATTC'}, 'expected_clean': {'CustomerID': 'RATTC'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10366}, 'cleaned_values': {'CustomerID': 'GALED'}, 'expected_clean': {'CustomerID': 'GALED'}, 'matche

## Problem 10: Employee assignment bị xoay vòng

**Vị trí:** `Orders.EmployeeID`  
**Loại lỗi:** `entity_relationship_mismatch`  
**Ground truth:** 987 assertions trên 987 rows

**Problem.** Các EmployeeID đều tồn tại và phân bố vẫn hợp lý, nhưng từng order đã được gán sai nhân viên.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [22]:
# Detect Problem 10
problem_10_candidates = show_problem_examples("T3-PLAUSIBLE-EMPLOYEE-REASSIGNMENT")

Candidate rows/groups: 16,282
Exact assertions: 987
Exact affected rows: 987
Verified raw examples:
  {'primary_key': {'OrderID': 10256}, 'raw_values': {'EmployeeID': 5}}
  {'primary_key': {'OrderID': 10261}, 'raw_values': {'EmployeeID': 9}}
  {'primary_key': {'OrderID': 10266}, 'raw_values': {'EmployeeID': 7}}
  {'primary_key': {'OrderID': 10295}, 'raw_values': {'EmployeeID': 9}}
  {'primary_key': {'OrderID': 10327}, 'raw_values': {'EmployeeID': 7}}


### Solution 10

Đây là contextual problem; exact historical assignment phải dùng oracle labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [23]:
def clean_plausible_employee_reassignment():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-PLAUSIBLE-EMPLOYEE-REASSIGNMENT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'All IDs are valid and distributionally plausible; exact assignment requires oracle labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_10_report = clean_plausible_employee_reassignment()
show_repaired_examples("T3-PLAUSIBLE-EMPLOYEE-REASSIGNMENT")

{'rule': 'T3-PLAUSIBLE-EMPLOYEE-REASSIGNMENT', 'method': 'supervised_ground_truth', 'reason': 'All IDs are valid and distributionally plausible; exact assignment requires oracle labels.', 'before': 987, 'updated': 987, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10256}, 'cleaned_values': {'EmployeeID': 3}, 'expected_clean': {'EmployeeID': 3}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10261}, 'cleaned_values': {'EmployeeID': 4}, 'expected_clean': {'EmployeeID': 4}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10266}, 'cleaned_values': {'EmployeeID': 3}, 'expected_clean': {'EmployeeID': 3}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10295}, 'cleaned_values': {'EmployeeID': 2}, 'expected_clean': {'EmployeeID': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10327}, 'cleaned_values': {'EmployeeID': 2}, 'expected_clean': {'EmployeeID': 2}, 'matches_ground_truth': True}


## Problem 11: Shipper assignment bị xoay vòng

**Vị trí:** `Orders.ShipVia`  
**Loại lỗi:** `entity_relationship_mismatch`  
**Ground truth:** 709 assertions trên 709 rows

**Problem.** ShipVia vẫn là ID hợp lệ nhưng shipper cụ thể của từng order đã thay đổi.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [24]:
# Detect Problem 11
problem_11_candidates = show_problem_examples("T3-PLAUSIBLE-SHIPPER-REASSIGNMENT")

Candidate rows/groups: 16,282
Exact assertions: 709
Exact affected rows: 709
Verified raw examples:
  {'primary_key': {'OrderID': 10250}, 'raw_values': {'ShipVia': 1}}
  {'primary_key': {'OrderID': 10254}, 'raw_values': {'ShipVia': 1}}
  {'primary_key': {'OrderID': 10260}, 'raw_values': {'ShipVia': 2}}
  {'primary_key': {'OrderID': 10276}, 'raw_values': {'ShipVia': 1}}
  {'primary_key': {'OrderID': 10312}, 'raw_values': {'ShipVia': 1}}


### Solution 11

Phân tích assignment pattern để phát hiện; phục hồi chính xác bằng supervised labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [25]:
def clean_plausible_shipper_reassignment():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-PLAUSIBLE-SHIPPER-REASSIGNMENT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'All shipper IDs are valid; exact historical assignment requires oracle labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_11_report = clean_plausible_shipper_reassignment()
show_repaired_examples("T3-PLAUSIBLE-SHIPPER-REASSIGNMENT")

{'rule': 'T3-PLAUSIBLE-SHIPPER-REASSIGNMENT', 'method': 'supervised_ground_truth', 'reason': 'All shipper IDs are valid; exact historical assignment requires oracle labels.', 'before': 709, 'updated': 709, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10250}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10254}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10260}, 'cleaned_values': {'ShipVia': 1}, 'expected_clean': {'ShipVia': 1}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10276}, 'cleaned_values': {'ShipVia': 3}, 'expected_clean': {'ShipVia': 3}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10312}, 'cleaned_values': {'ShipVia': 2}, 'expected_clean': {'ShipVia': 2}, 'matches_ground_truth': True}


## Problem 12: OrderDate bị dịch đúng 365 ngày

**Vị trí:** `Orders.OrderDate`  
**Loại lỗi:** `temporal_context_anomaly`  
**Ground truth:** 1,000 assertions trên 1,000 rows

**Problem.** OrderDate vẫn là ISO timestamp hợp lệ nhưng lệch khỏi RequiredDate, ShippedDate và temporal distribution.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [26]:
# Detect Problem 12
problem_12_candidates = show_problem_examples("T3-ONE-YEAR-DATE-SHIFT")

Candidate rows/groups: 1,000
Exact assertions: 1,000
Exact affected rows: 1,000
Verified raw examples:
  {'primary_key': {'OrderID': 10256}, 'raw_values': {'OrderDate': '2017-07-15 00:00:00'}}
  {'primary_key': {'OrderID': 10257}, 'raw_values': {'OrderDate': '2017-07-16 00:00:00'}}
  {'primary_key': {'OrderID': 10283}, 'raw_values': {'OrderDate': '2017-08-16 00:00:00'}}
  {'primary_key': {'OrderID': 10290}, 'raw_values': {'OrderDate': '2017-08-27 00:00:00'}}
  {'primary_key': {'OrderID': 10297}, 'raw_values': {'OrderDate': '2017-09-04 00:00:00'}}


### Solution 12

Trừ chính xác 365 ngày và bảo toàn độ chính xác date/time của clean label.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [27]:
def clean_one_year_date_shift():
    """Reverse the generator's exact 365-day shift."""
    rule_id = "T3-ONE-YEAR-DATE-SHIFT"

    def transform_raw_value(value, record):
        parsed = datetime.strptime(str(value), "%Y-%m-%d %H:%M:%S")
        repaired = parsed - timedelta(days=365)
        label = str(record["clean_value"])
        if " " in label:
            return repaired.strftime("%Y-%m-%d %H:%M:%S")
        return repaired.date().isoformat()

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_12_report = clean_one_year_date_shift()
show_repaired_examples("T3-ONE-YEAR-DATE-SHIFT")

{'rule': 'T3-ONE-YEAR-DATE-SHIFT', 'before': 1000, 'updated': 1000, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10256}, 'cleaned_values': {'OrderDate': '2016-07-15'}, 'expected_clean': {'OrderDate': '2016-07-15'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10257}, 'cleaned_values': {'OrderDate': '2016-07-16'}, 'expected_clean': {'OrderDate': '2016-07-16'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10283}, 'cleaned_values': {'OrderDate': '2016-08-16'}, 'expected_clean': {'OrderDate': '2016-08-16'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10290}, 'cleaned_values': {'OrderDate': '2016-08-27'}, 'expected_clean': {'OrderDate': '2016-08-27'}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10297}, 'cleaned_values': {'OrderDate': '2016-09-04'}, 'expected_clean': {'OrderDate': '2016-09-04'}, 'matches_ground_truth': True}


## Problem 13: Freight bị sai thang đo 1/100

**Vị trí:** `Orders.Freight`  
**Loại lỗi:** `unit_scale_mismatch`  
**Ground truth:** 800 assertions trên 800 rows

**Problem.** Freight vẫn dương và có thể trông hợp lệ, nhưng một nhóm giá trị đã bị chia 100.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [28]:
# Detect Problem 13
problem_13_candidates = show_problem_examples("T3-FREIGHT-SCALE-DRIFT")

Candidate rows/groups: 800
Exact assertions: 800
Exact affected rows: 800
Verified raw examples:
  {'primary_key': {'OrderID': 10294}, 'raw_values': {'Freight': 0.28750000000000003}}
  {'primary_key': {'OrderID': 10304}, 'raw_values': {'Freight': 0.20500000000000002}}
  {'primary_key': {'OrderID': 10315}, 'raw_values': {'Freight': 0.21}}
  {'primary_key': {'OrderID': 10324}, 'raw_values': {'Freight': 0.7025}}
  {'primary_key': {'OrderID': 10342}, 'raw_values': {'Freight': 0.5}}


### Solution 13

Phát hiện scale cluster theo context, nhân affected values với 100 và làm tròn về precision chuẩn.

**Solution mode:** `deterministic_transform`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [29]:
def clean_freight_scale_drift():
    """Reverse the injected 1/100 unit-scale drift."""
    rule_id = "T3-FREIGHT-SCALE-DRIFT"

    def transform_raw_value(value, record):
        number = round(float(value) * 100, 10)
        return int(number) if number.is_integer() else number

    # apply_transform reads current raw values, calculates repaired values,
    # checks every result against clean labels, and performs parameterized UPDATEs.
    return apply_transform(rule_id, transform_raw_value)

problem_13_report = clean_freight_scale_drift()
show_repaired_examples("T3-FREIGHT-SCALE-DRIFT")

{'rule': 'T3-FREIGHT-SCALE-DRIFT', 'before': 800, 'updated': 800, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10294}, 'cleaned_values': {'Freight': 28.75}, 'expected_clean': {'Freight': 28.75}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10304}, 'cleaned_values': {'Freight': 20.5}, 'expected_clean': {'Freight': 20.5}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10315}, 'cleaned_values': {'Freight': 21}, 'expected_clean': {'Freight': 21}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10324}, 'cleaned_values': {'Freight': 70.25}, 'expected_clean': {'Freight': 70.25}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10342}, 'cleaned_values': {'Freight': 50}, 'expected_clean': {'Freight': 50}, 'matches_ground_truth': True}


## Problem 14: Order-line price hợp lệ nhưng thuộc dòng khác

**Vị trí:** `Order Details.UnitPrice`  
**Loại lỗi:** `contextual_value_mismatch`  
**Ground truth:** 1,799 assertions trên 1,799 rows

**Problem.** UnitPrice được hoán vị giữa order lines nên vẫn đúng kiểu và global range. Giá lịch sử cũng có thể khác current product price.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [30]:
# Detect Problem 14
problem_14_candidates = show_problem_examples("T3-CONTEXTUAL-DETAIL-PRICE")

Candidate rows/groups: 475,277
Exact assertions: 1,799
Exact affected rows: 1,799
Verified raw examples:
  {'primary_key': {'OrderID': 10253, 'ProductID': 39}, 'raw_values': {'UnitPrice': 21}}
  {'primary_key': {'OrderID': 10342, 'ProductID': 55}, 'raw_values': {'UnitPrice': 19.45}}
  {'primary_key': {'OrderID': 10549, 'ProductID': 45}, 'raw_values': {'UnitPrice': 38}}
  {'primary_key': {'OrderID': 10716, 'ProductID': 61}, 'raw_values': {'UnitPrice': 18}}
  {'primary_key': {'OrderID': 10813, 'ProductID': 2}, 'raw_values': {'UnitPrice': 7.45}}


### Solution 14

Dùng product/time context để tạo candidate; exact historical line price cần supervised labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [31]:
def clean_contextual_detail_price():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-CONTEXTUAL-DETAIL-PRICE"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Historical prices can legitimately differ from current product price; exact line price requires labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_14_report = clean_contextual_detail_price()
show_repaired_examples("T3-CONTEXTUAL-DETAIL-PRICE")

{'rule': 'T3-CONTEXTUAL-DETAIL-PRICE', 'method': 'supervised_ground_truth', 'reason': 'Historical prices can legitimately differ from current product price; exact line price requires labels.', 'before': 1799, 'updated': 1799, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10253, 'ProductID': 39}, 'cleaned_values': {'UnitPrice': 14.4}, 'expected_clean': {'UnitPrice': 14.4}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10342, 'ProductID': 55}, 'cleaned_values': {'UnitPrice': 19.2}, 'expected_clean': {'UnitPrice': 19.2}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10549, 'ProductID': 45}, 'cleaned_values': {'UnitPrice': 9.5}, 'expected_clean': {'UnitPrice': 9.5}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10716, 'ProductID': 61}, 'cleaned_values': {'UnitPrice': 28.5}, 'expected_clean': {'UnitPrice': 28.5}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10813, 'ProductID': 2}, 'cleaned_values': {'UnitPrice': 1

## Problem 15: Quantity hợp lệ nhưng bị hoán vị giữa các dòng

**Vị trí:** `Order Details.Quantity`  
**Loại lỗi:** `contextual_value_mismatch`  
**Ground truth:** 1,499 assertions trên 1,499 rows

**Problem.** Các quantity vẫn nằm trong miền và giữ gần nguyên distribution, nhưng sai ở cấp order line.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [32]:
# Detect Problem 15
problem_15_candidates = show_problem_examples("T3-CONTEXTUAL-DETAIL-QUANTITY")

Candidate rows/groups: 77
Exact assertions: 1,499
Exact affected rows: 1,499
Verified raw examples:
  {'primary_key': {'OrderID': 10729, 'ProductID': 21}, 'raw_values': {'Quantity': 33}}
  {'primary_key': {'OrderID': 10742, 'ProductID': 3}, 'raw_values': {'Quantity': 7}}
  {'primary_key': {'OrderID': 11083, 'ProductID': 68}, 'raw_values': {'Quantity': 33}}
  {'primary_key': {'OrderID': 11090, 'ProductID': 74}, 'raw_values': {'Quantity': 36}}
  {'primary_key': {'OrderID': 11091, 'ProductID': 68}, 'raw_values': {'Quantity': 23}}


### Solution 15

Univariate profiling chỉ cho dấu hiệu; exact row quantities phải được phục hồi từ labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [33]:
def clean_contextual_detail_quantity():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-CONTEXTUAL-DETAIL-QUANTITY"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Permuted quantities preserve valid range and distribution; exact row values require labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_15_report = clean_contextual_detail_quantity()
show_repaired_examples("T3-CONTEXTUAL-DETAIL-QUANTITY")

{'rule': 'T3-CONTEXTUAL-DETAIL-QUANTITY', 'method': 'supervised_ground_truth', 'reason': 'Permuted quantities preserve valid range and distribution; exact row values require labels.', 'before': 1499, 'updated': 1499, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10729, 'ProductID': 21}, 'cleaned_values': {'Quantity': 30}, 'expected_clean': {'Quantity': 30}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10742, 'ProductID': 3}, 'cleaned_values': {'Quantity': 20}, 'expected_clean': {'Quantity': 20}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11083, 'ProductID': 68}, 'cleaned_values': {'Quantity': 9}, 'expected_clean': {'Quantity': 9}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11090, 'ProductID': 74}, 'cleaned_values': {'Quantity': 15}, 'expected_clean': {'Quantity': 15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 11091, 'ProductID': 68}, 'cleaned_values': {'Quantity': 36}, 'expected_clean': {'Quantity':

## Problem 16: Discount level hợp lệ nhưng sai context

**Vị trí:** `Order Details.Discount`  
**Loại lỗi:** `contextual_value_mismatch`  
**Ground truth:** 605 assertions trên 605 rows

**Problem.** Mỗi discount vẫn nằm trong tập mức hợp lệ, nhưng label đã được gán sang order line khác.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [34]:
# Detect Problem 16
problem_16_candidates = show_problem_examples("T3-DISCOUNT-LABEL-DRIFT")

Candidate rows/groups: 838
Exact assertions: 605
Exact affected rows: 605
Verified raw examples:
  {'primary_key': {'OrderID': 10250, 'ProductID': 51}, 'raw_values': {'Discount': 0.2}}
  {'primary_key': {'OrderID': 10250, 'ProductID': 65}, 'raw_values': {'Discount': 0.2}}
  {'primary_key': {'OrderID': 10251, 'ProductID': 22}, 'raw_values': {'Discount': 0.15}}
  {'primary_key': {'OrderID': 10251, 'ProductID': 57}, 'raw_values': {'Discount': 0.15}}
  {'primary_key': {'OrderID': 10252, 'ProductID': 20}, 'raw_values': {'Discount': 0.1}}


### Solution 16

Kiểm tra discount theo order context và dùng supervised labels cho exact repair.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [35]:
def clean_discount_label_drift():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-DISCOUNT-LABEL-DRIFT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Every discount remains valid; exact contextual discount requires supervised labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_16_report = clean_discount_label_drift()
show_repaired_examples("T3-DISCOUNT-LABEL-DRIFT")

{'rule': 'T3-DISCOUNT-LABEL-DRIFT', 'method': 'supervised_ground_truth', 'reason': 'Every discount remains valid; exact contextual discount requires supervised labels.', 'before': 605, 'updated': 605, 'after': 0}
After cleaning examples:
  {'primary_key': {'OrderID': 10250, 'ProductID': 51}, 'cleaned_values': {'Discount': 0.15}, 'expected_clean': {'Discount': 0.15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10250, 'ProductID': 65}, 'cleaned_values': {'Discount': 0.15}, 'expected_clean': {'Discount': 0.15}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10251, 'ProductID': 22}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Discount': 0.05}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10251, 'ProductID': 57}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Discount': 0.05}, 'matches_ground_truth': True}
  {'primary_key': {'OrderID': 10252, 'ProductID': 20}, 'cleaned_values': {'Discount': 0.05}, 'expected_clean': {'Disco

## Problem 17: ContactName và ContactTitle thuộc sai customer

**Vị trí:** `Customers.ContactName, ContactTitle`  
**Loại lỗi:** `multi_column_entity_mismatch`  
**Ground truth:** 119 assertions trên 60 rows

**Problem.** Tên và chức danh đều có vẻ hợp lệ nhưng contact pair đã được chuyển giữa các customer entities.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [36]:
# Detect Problem 17
problem_17_candidates = show_problem_examples("T3-CUSTOMER-CONTACT-MISALIGNMENT")

Candidate rows/groups: 93
Exact assertions: 119
Exact affected rows: 60
Verified raw examples:
  {'primary_key': {'CustomerID': 'ALFKI'}, 'raw_values': {'ContactName': 'Maria Larsson', 'ContactTitle': 'Owner'}}
  {'primary_key': {'CustomerID': 'ANATR'}, 'raw_values': {'ContactName': 'Carine Schmitt', 'ContactTitle': 'Marketing Manager'}}
  {'primary_key': {'CustomerID': 'ANTON'}, 'raw_values': {'ContactName': 'Eduardo Saavedra', 'ContactTitle': 'Marketing Manager'}}
  {'primary_key': {'CustomerID': 'AROUT'}, 'raw_values': {'ContactName': 'José Pedro Freyre', 'ContactTitle': 'Sales Manager'}}
  {'primary_key': {'CustomerID': 'BERGS'}, 'raw_values': {'ContactName': 'Howard Snyder', 'ContactTitle': 'Marketing Manager'}}


### Solution 17

Validate identity fields như một nhóm và phục hồi cả pair bằng ground truth.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [37]:
def clean_customer_contact_misalignment():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-CUSTOMER-CONTACT-MISALIGNMENT"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Plausible contact pairs were moved between entities; exact identity requires labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_17_report = clean_customer_contact_misalignment()
show_repaired_examples("T3-CUSTOMER-CONTACT-MISALIGNMENT")

{'rule': 'T3-CUSTOMER-CONTACT-MISALIGNMENT', 'method': 'supervised_ground_truth', 'reason': 'Plausible contact pairs were moved between entities; exact identity requires labels.', 'before': 119, 'updated': 119, 'after': 0}
After cleaning examples:
  {'primary_key': {'CustomerID': 'ALFKI'}, 'cleaned_values': {'ContactName': 'Maria Anders', 'ContactTitle': 'Sales Representative'}, 'expected_clean': {'ContactName': 'Maria Anders', 'ContactTitle': 'Sales Representative'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'ANATR'}, 'cleaned_values': {'ContactName': 'Ana Trujillo', 'ContactTitle': 'Owner'}, 'expected_clean': {'ContactName': 'Ana Trujillo', 'ContactTitle': 'Owner'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'ANTON'}, 'cleaned_values': {'ContactName': 'Antonio Moreno', 'ContactTitle': 'Owner'}, 'expected_clean': {'ContactName': 'Antonio Moreno', 'ContactTitle': 'Owner'}, 'matches_ground_truth': True}
  {'primary_key': {'CustomerID': 'AROUT'}, 

## Problem 18: Product price hợp lệ nhưng thuộc sản phẩm khác

**Vị trí:** `Products.UnitPrice`  
**Loại lỗi:** `contextual_value_mismatch`  
**Ground truth:** 60 assertions trên 60 rows

**Problem.** Giá sản phẩm được xoay vòng nên global distribution không đổi nhưng từng product mang giá sai.

Chạy cell Detect để thực thi candidate SQL và xem verified raw examples. Candidate có thể rộng hơn exact faults, đặc biệt với contextual problems.

In [38]:
# Detect Problem 18
problem_18_candidates = show_problem_examples("T3-PLAUSIBLE-PRODUCT-PRICE")

Candidate rows/groups: 77
Exact assertions: 60
Exact affected rows: 60
Verified raw examples:
  {'primary_key': {'ProductID': 11}, 'raw_values': {'UnitPrice': 32}}
  {'primary_key': {'ProductID': 13}, 'raw_values': {'UnitPrice': 2.5}}
  {'primary_key': {'ProductID': 15}, 'raw_values': {'UnitPrice': 14}}
  {'primary_key': {'ProductID': 16}, 'raw_values': {'UnitPrice': 18}}
  {'primary_key': {'ProductID': 17}, 'raw_values': {'UnitPrice': 19}}


### Solution 18

Đối chiếu product identity và lịch sử; exact UnitPrice được phục hồi từ labels.

**Solution mode:** `oracle_required`. Sau khi chạy, output hiển thị 5 examples và `matches_ground_truth` để kiểm tra trực tiếp.

In [39]:
def clean_plausible_product_price():
    """Restore exact labels because raw candidates do not identify one unique clean value."""
    rule_id = "T3-PLAUSIBLE-PRODUCT-PRICE"
    before = unresolved_count(rule_id)
    updated = 0

    # Each label supplies table, column, clean value, and row identity.
    for record in records_for(rule_id):
        # A corrupted primary key must locate the raw row before restoring the key.
        locator = current_locator(record)
        where, parameters = primary_key_filter(locator)
        update_sql = (
            f"UPDATE {quote_name(record['table'])} "
            f"SET {quote_name(record['column'])} = ? WHERE {where}"
        )
        cursor = connection.execute(
            update_sql, [record["clean_value"], *parameters]
        )
        assert cursor.rowcount == 1, (rule_id, locator)
        updated += 1

    report = {
        "rule": rule_id,
        "method": "supervised_ground_truth",
        "reason": 'Prices were permuted while preserving distribution; exact product price requires labels.',
        "before": before,
        "updated": updated,
        "after": unresolved_count(rule_id),
    }
    print(report)
    return report

problem_18_report = clean_plausible_product_price()
show_repaired_examples("T3-PLAUSIBLE-PRODUCT-PRICE")

{'rule': 'T3-PLAUSIBLE-PRODUCT-PRICE', 'method': 'supervised_ground_truth', 'reason': 'Prices were permuted while preserving distribution; exact product price requires labels.', 'before': 60, 'updated': 60, 'after': 0}
After cleaning examples:
  {'primary_key': {'ProductID': 11}, 'cleaned_values': {'UnitPrice': 21}, 'expected_clean': {'UnitPrice': 21}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 13}, 'cleaned_values': {'UnitPrice': 6}, 'expected_clean': {'UnitPrice': 6}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 15}, 'cleaned_values': {'UnitPrice': 15.5}, 'expected_clean': {'UnitPrice': 15.5}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 16}, 'cleaned_values': {'UnitPrice': 17.45}, 'expected_clean': {'UnitPrice': 17.45}, 'matches_ground_truth': True}
  {'primary_key': {'ProductID': 17}, 'cleaned_values': {'UnitPrice': 39}, 'expected_clean': {'UnitPrice': 39}, 'matches_ground_truth': True}


## Final validation

Tất cả faults phải được repair, clean controls không được thay đổi, foreign keys phải hợp lệ và 13 table fingerprints phải khớp expected clean state trong JSON.

In [40]:
def primary_key_columns(table):
    rows = connection.execute(f"PRAGMA table_info({quote_name(table)})").fetchall()
    return [row["name"] for row in sorted(
        (row for row in rows if row["pk"]), key=lambda row: row["pk"])]

def table_fingerprint(table):
    keys = primary_key_columns(table)
    query = f"SELECT * FROM {quote_name(table)}"
    if keys:
        query += " ORDER BY " + ", ".join(quote_name(key) for key in keys)
    digest = hashlib.sha256()
    for row in connection.execute(query):
        values = [{"blob_sha256": hashlib.sha256(value).hexdigest(), "bytes": len(value)}
                  if isinstance(value, bytes) else value for value in row]
        line = json.dumps(values, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
        digest.update(line.encode("utf-8") + b"\n")
    return digest.hexdigest()

def score_ground_truth():
    repaired = remaining = preserved = false_positive = 0
    for record in bundle["repair_records"]:
        row = find_row(record["table"], record["primary_key"])
        if record["operation"] == "insert_row":
            repaired += int(row is None); remaining += int(row is not None)
        elif row is not None and row[record["column"]] == record["clean_value"]:
            repaired += 1
        else:
            remaining += 1
    for control in bundle["clean_controls"]:
        row = find_row(control["table"], control["primary_key"])
        if row is not None and row[control["column"]] == control["clean_value"]:
            preserved += 1
        else:
            false_positive += 1
    total = repaired + remaining + preserved + false_positive
    return {"quality_score": round(100 * (repaired + preserved) / total, 4),
            "repaired_faults": repaired, "remaining_faults": remaining,
            "preserved_clean_controls": preserved,
            "false_positive_controls": false_positive}

connection.execute("DELETE FROM sqlite_sequence")
connection.executemany("INSERT INTO sqlite_sequence(name, seq) VALUES (?, ?)",
    [(item["name"], item["seq"])
     for item in bundle["validation_expectations"]["sqlite_sequence"]])
connection.commit()

expected = bundle["validation_expectations"]
matching_tables = 0
for table, wanted in expected["tables"].items():
    rows = connection.execute(f"SELECT COUNT(*) FROM {quote_name(table)}").fetchone()[0]
    matching_tables += int(rows == wanted["rows"]
                           and table_fingerprint(table) == wanted["fingerprint"])
sequence = [dict(row) for row in connection.execute(
    "SELECT name, seq FROM sqlite_sequence ORDER BY name")]
final_validation = {
    **score_ground_truth(),
    "integrity_check": connection.execute("PRAGMA integrity_check").fetchone()[0],
    "foreign_key_violations": len(connection.execute("PRAGMA foreign_key_check").fetchall()),
    "matching_tables": matching_tables,
    "table_count": len(expected["tables"]),
    "sqlite_sequence_match": sequence == expected["sqlite_sequence"],
}
final_validation["passed"] = (
    final_validation["quality_score"] == 100.0
    and final_validation["remaining_faults"] == 0
    and final_validation["false_positive_controls"] == 0
    and final_validation["integrity_check"] == "ok"
    and final_validation["foreign_key_violations"] == 0
    and matching_tables == len(expected["tables"])
    and final_validation["sqlite_sequence_match"])
print(json.dumps(final_validation, indent=2))
assert final_validation["passed"]

{
  "quality_score": 100.0,
  "repaired_faults": 14052,
  "remaining_faults": 0,
  "preserved_clean_controls": 9368,
  "false_positive_controls": 0,
  "integrity_check": "ok",
  "foreign_key_violations": 0,
  "matching_tables": 13,
  "table_count": 13,
  "sqlite_sequence_match": true,
  "passed": true
}


## Publish `northwind_cleaned3.db`

Chỉ sau khi validation pass, working database mới được ghi ra clean output. Raw database không bị thay đổi.

In [41]:
descriptor, temporary_name = tempfile.mkstemp(
    prefix=".notebook_clean_", suffix=".db", dir=FOLDER)
os.close(descriptor)
temporary_path = Path(temporary_name)
try:
    destination = sqlite3.connect(temporary_path)
    connection.backup(destination)
    destination.close()
    os.replace(temporary_path, CLEAN)
finally:
    temporary_path.unlink(missing_ok=True)
print("Published:", CLEAN.name)
print("Output SHA-256:", sha256_file(CLEAN))
print("Raw remained unchanged:", sha256_file(RAW) == bundle["dataset"]["raw_sha256"])
connection.close()

Published: northwind_cleaned3.db
Output SHA-256: 9290f09fba08ab0a4e5b15289785b9353d2dcfcedc4cb6a36b6b9c27d83c33fe
Raw remained unchanged: True


## Hoàn thành

JSON là machine-readable knowledge, notebook là tài liệu problem/solution trực quan, và Python là executable reference answer end-to-end.